# M59e — first encoder internals
Revision: **M59e-2026-09-18-r1**. Run the three code cells below in order. **CPU is sufficient; no training.**

This notebook replays the trusted, hash-verified M59d TorchScript you already exported. No MonoDGP checkout, KITTI download, CUDA extension, or checkpoint reconstruction is needed. It does not repair the model or evaluate AP. Export success must be followed by the macOS runtime comparison.

The 2026-09-18 local run has already identified the fourth positional scale's sine/cosine channel-order mismatch. This notebook is for reproducing that diagnostic, not a required new training run.

In [ ]:
# Cell 1 — self-contained paths and durable logging. M59e-2026-09-18-r1
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REVISION = 'M59e-2026-09-18-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
MOBILE_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
COMPRESSION_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression')
M59D_DIR = COMPRESSION_ROOT / 'monodgp_m59d_2d_transformer'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = COMPRESSION_ROOT / 'monodgp_m59e_encoder_internals' / 'runs' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
LOG_DIR = OUTPUT_DIR / 'colab_logs'
LOG_DIR.mkdir()
def run_logged(command, cwd, name):
    command = [str(value) for value in command]
    log_path = LOG_DIR / (name + '.log')
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=40)
    with log_path.open('w') as log:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            code = process.wait()
    if code:
        raise RuntimeError(f'Exit {code}; log={log_path}\n' + '\n'.join(tail))
    return log_path
print('Revision:', REVISION, '\nFrozen input:', M59D_DIR, '\nNew output:', OUTPUT_DIR)


In [ ]:
# Cell 2 — sync our main branch and install CPU export dependencies.
if not MOBILE_REPO.exists():
    run_logged(['git', 'clone', '--branch', 'main', MOBILE_URL, MOBILE_REPO], None, 'clone_mobile')
else:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected mobile_adas3d main, found {branch!r}; preserve local changes before switching.')
    run_logged(['git', 'pull', '--ff-only', 'origin', 'main'], MOBILE_REPO, 'update_mobile')
run_logged([sys.executable, '-m', 'pip', 'install', 'coremltools==9.0', 'numpy>=2.0,<2.4'], MOBILE_REPO, 'dependencies')
SCRIPT = MOBILE_REPO / 'scripts/export_monodgp_m59e_encoder_internals.py'
required = [SCRIPT] + [M59D_DIR / name for name in (
    'm59d_2d_transformer_export_gate.json',
    'MonoDGP_M59d_2d_transformer_fp32.pt',
    'm59d_2d_transformer_reference_io.npz',
)]
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Required M59d export inputs missing:\n' + '\n'.join(missing))
run_logged([sys.executable, '-c', 'import torch, numpy, coremltools; print(torch.__version__, numpy.__version__, coremltools.__version__)'], MOBILE_REPO, 'versions')
print('Project revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MOBILE_REPO, text=True).strip())


In [ ]:
# Cell 3 — instrument the frozen trace, verify preservation, export, and bundle.
# A new output directory is required for each attempt; rerun from Cell 1 after a failed export.
run_logged([sys.executable, '-u', SCRIPT, '--m59d-dir', M59D_DIR, '--output-dir', OUTPUT_DIR], MOBILE_REPO, 'm59e_export')
GATE_PATH = OUTPUT_DIR / 'm59e_encoder_internal_export_gate.json'
gate = json.loads(GATE_PATH.read_text())
assert gate['complete'] and gate['all_export_gates_passed']
assert not gate['training_performed'] and not gate['weights_changed']
ARCHIVE = shutil.make_archive(str(OUTPUT_DIR.parent / (RUN_ID + '_m59e')), 'zip', root_dir=OUTPUT_DIR)
print('Export PASS (macOS runtime parity still required).')
print('Send this archive:', ARCHIVE)
print('Export report:', GATE_PATH)


## On the Mac — compare actual Core ML execution
Extract the archive into a new local directory. From the project checkout, run `scripts/validate_monodgp_m59e_macos.py` with `--artifact-dir`, `--compute-units ALL`, `--output`, and `--save-predictions`. The command intentionally exits nonzero if parity fails, but still writes a complete diagnostic report. See `MONODGP_M59E_ENCODER_INTERNALS_CONTRACT.md` for exact commands and the optional channel-layout analysis.

**Stop after the diagnostic.** No full evaluation, retraining, precision reduction, or iPhone test is authorized by this notebook.